In [ ]:
from rag_helper import RAGBase
from openai import OpenAI

In [ ]:
import os

In [ ]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

### Asking without tools

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
)

response.output_text

### Defining the tool

In [ ]:
from ingest import load_faq_data, build_index

In [ ]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")
index = build_index(documents)

In [ ]:
def search(query):
   boost_dict = {"question": 2.0, "section": 0.5}
   filter_dict = {"course": "llm-zoomcamp"}

   return index.search(
     query, 
     boost_dict=boost_dict,
     filter_dict=filter_dict,
     num_results=5
  )

In [ ]:
search_results = search("How do I run Ollama?")
search_results

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [ ]:
import json

# 1) Check messages serializability
try:
    json_str = json.dumps(messages)
    print("messages OK (len bytes):", len(json_str))
except Exception as e:
    print("messages not serializable:", type(e).__name__, e)

# 2) Check tools/search_tool serializability
try:
    json_str = json.dumps([search_tool])
    print("tools OK (len bytes):", len(json_str))
except Exception as e:
    print("tools not serializable:", type(e).__name__, e)

# 3) Inspect contents briefly
print("messages preview:", messages[:2])
print("search_tool preview keys:", list(search_tool.keys()))

In [ ]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output

### Executing the function and sending the result back

In [ ]:
import json
call = response.output[1]
call
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [ ]:
messages.append(response.output[1])

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
messages

In [ ]:
import json

# 1) Check messages serializability
try:
    json_str = json.dumps(messages)
    print("messages OK (len bytes):", len(json_str))
except Exception as e:
    print("messages not serializable:", type(e).__name__, e)

# 2) Check tools/search_tool serializability
try:
    json_str = json.dumps([search_tool])
    print("tools OK (len bytes):", len(json_str))
except Exception as e:
    print("tools not serializable:", type(e).__name__, e)

# 3) Inspect contents briefly
print("messages preview:", messages[:2])
print("search_tool preview keys:", list(search_tool.keys()))

In [ ]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output_text